# Laboratório 05 — Classificação de ponta a ponta

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

No lab04 os dados eram **quarenta apartamentos inventados por um gerador**. Hoje
são **20.640 distritos da Califórnia**, do censo de 1990 — os mesmos do capítulo 2
do Géron. Dados de verdade vêm com tudo que dado de verdade tem: valores faltando,
coluna de texto, escalas incompatíveis e um limite artificial no alvo.

O roteiro é o do capítulo 2, com uma diferença: em vez de **prever o preço**, vamos
**classificar o distrito em caro / não caro**. Não é liberdade nossa; é uma
possibilidade que o próprio livro levanta, na p. 42:

> *"e se o sistema downstream converter os preços em categorias (por exemplo,
> 'barato', 'médio' ou 'caro') [...]? Neste caso, não é importante saber o preço
> exato se o seu sistema só precisa obter a categoria certa. Se for assim, então o
> problema deveria ter sido enquadrado como uma **tarefa de classificação**, não uma
> tarefa de regressão."*

É exatamente o nosso caso hoje. E assim o laboratório junta as duas aulas: o
**ponta a ponta** do capítulo 2 e as **métricas de classificação** do capítulo 3.

O que você vai fazer, em ordem:

| Parte | O que |
|---|---|
| 0–1 | carregar os dados e **enquadrar** o problema |
| 2 | separar treino e teste **antes de olhar** |
| 3 | explorar — **só o treino** |
| 4 | a linha de base burra, e por que 75% de acurácia não vale nada |
| 5 | preparar: faltantes, texto, escala — num `Pipeline` |
| 6 | **a armadilha do vazamento** |
| 7–9 | logística, árvore, floresta |
| 10 | o limiar |
| 11 | a avaliação final, **uma vez só** |
| 12 | análise de erros: **onde** ele erra |

> ⚙️ Precisa de `numpy`, `pandas`, `matplotlib` e `scikit-learn`.

> 💻 **Roda em qualquer lugar.** Se o `housing.csv` não estiver na pasta, a célula
> da Parte 0 baixa sozinha. Funciona no laboratório, em casa e no Google Colab.

> ⚠️ **Atenção ao livro de vocês.** O PDF do Géron que está na pasta da disciplina é
> a edição de **2019**, e o código do capítulo 2 dela **não roda** no Scikit-Learn de
> hoje: ele usa `Imputer` e `CategoricalEncoder`, que não existem mais. Os nomes
> atuais são `SimpleImputer` (em `sklearn.impute`, não em `preprocessing`) e
> `OneHotEncoder`. Este notebook usa os nomes atuais.

## Parte 0 — Os dados

Rode a célula. Se o `housing.csv` não estiver aqui, ela baixa do repositório do
próprio Géron e grava na pasta — da segunda vez em diante já é local.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 20260929
URL = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
       "master/datasets/housing/housing.csv")

arquivo = Path("housing.csv")
if not arquivo.exists():
    print("baixando housing.csv ...")
    pd.read_csv(URL).to_csv(arquivo, index=False)

dados = pd.read_csv(arquivo)


def reais(v):
    """Formata no padrão brasileiro: 179700 -> 'R$ 179.700'."""
    return "R$ " + f"{v:,.0f}".replace(",", ".")


print(f"{len(dados)} distritos, {len(dados.columns)} colunas")
dados.head()

Agora o retrato honesto do conjunto. **Leia a saída inteira antes de seguir** —
há três coisas erradas ali, e todas voltam mais adiante.

In [ ]:
dados.info()
print()
print(dados["ocean_proximity"].value_counts())
print()
print(dados["median_house_value"].describe())
print()
teto = (dados["median_house_value"] == 500001).sum()
print(f"distritos com median_house_value exatamente 500.001: {teto}")

**As três coisas.**

1. **`total_bedrooms` tem 20.433 valores, não 20.640.** Faltam 207. Quase todo
   algoritmo do Scikit-Learn quebra com `NaN` — vamos resolver na Parte 5.
2. **`ocean_proximity` é texto.** Cinco valores possíveis, e um deles (`ISLAND`)
   aparece **5 vezes** em vinte mil. Modelo nenhum come texto: vira número na Parte 5.
3. **O alvo está truncado.** O máximo de `median_house_value` é exatamente
   **500.001**, e há **965 distritos** nesse valor. Não é coincidência: quem montou
   o censo cortou ali. Quer dizer que um distrito de 700 mil e um de 500 mil estão
   registrados com o mesmo número.

> 💡 O item 3 é um argumento **a favor** de classificar em vez de prever o valor: a
> regressão tenta acertar um número que, no topo da faixa, simplesmente não é o
> número certo.

## Parte 1 — Enquadrar: de regressão para classificação

Nosso alvo será binário:

$$\text{caro} = \begin{cases} 1 & \text{se median\_house\_value} > 265.000 \\
0 & \text{caso contrário}\end{cases}$$

Por que 265 mil, e não a mediana? Porque um corte na mediana daria 50% de cada
classe, e a realidade raramente é assim. Com 265 mil ficamos com cerca de **um
quarto** de distritos caros — desbalanceado, como quase todo problema de
classificação útil. E desbalanceado é justamente onde a acurácia engana.

In [ ]:
CORTE = 265000.0

# TODO 1: crie a coluna "caro", valendo 1 quando median_house_value for MAIOR
#         que CORTE e 0 caso contrário.
#         Dica: (dados["median_house_value"] > CORTE).astype(int)
#         (a linha abaixo é provisória — corta na mediana, não em CORTE — e está
#          aqui só para o notebook rodar antes de você preencher)
dados["caro"] = (dados["median_house_value"]
                 > dados["median_house_value"].median()).astype(int)

print(dados["caro"].value_counts())
print(f"\nproporção de caros: {dados['caro'].mean():.4f}   (esperado: 0.2494)")

## Parte 2 — Separar treino e teste, antes de olhar

Esta é a primeira coisa a fazer, e o livro é enfático: separar **antes** de explorar.
O motivo tem nome — ***data snooping***. Se você olha o conjunto inteiro, acha um
padrão, e só então separa, o "teste" já não testa nada: você é que aprendeu com ele,
e passou o que aprendeu para o modelo na forma de escolhas suas.

E vamos separar de forma **estratificada**. Amostra aleatória simples, em 20.640
linhas, deixa a proporção de faixas de renda ligeiramente diferente entre treino e
teste — e a renda é o atributo mais correlacionado com o preço. Estratificar garante
que as duas metades tenham a mesma composição.

> 📖 É a seção *"Crie um Conjunto de Testes"* (p. 51 da edição de 2019). O código do
> livro usa `np.ceil(median_income / 1.5)`; aqui usamos `pd.cut`, que é o idioma
> atual e diz na cara quais são os cortes.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

dados["faixa_renda"] = pd.cut(dados["median_income"],
                              bins=[0., 1.5, 3., 4.5, 6., np.inf],
                              labels=[1, 2, 3, 4, 5])

# TODO 2: separe 20% para teste, estratificando por "faixa_renda".
#         Dica: divisor = StratifiedShuffleSplit(n_splits=1, test_size=0.2,
#                                                random_state=SEMENTE)
#               i_tr, i_te = next(divisor.split(dados, dados["faixa_renda"]))
#         (a linha abaixo é um corte provisório, NÃO estratificado, só para o
#          notebook rodar antes de você preencher)
i_tr, i_te = np.arange(0, 16512), np.arange(16512, 20640)

treino = dados.iloc[i_tr].drop(columns=["faixa_renda"]).reset_index(drop=True)
teste  = dados.iloc[i_te].drop(columns=["faixa_renda"]).reset_index(drop=True)

print(f"treino: {len(treino)}    teste: {len(teste)}")
print(f"caros — total {dados['caro'].mean():.4f}   "
      f"treino {treino['caro'].mean():.4f}   teste {teste['caro'].mean():.4f}")
print( "esperado:     0.2494              0.2489              0.2512")

> 🔒 **A partir daqui, `teste` não é tocado.** Ele volta uma única vez, na Parte 11.
> Se você espiar antes, o número final deixa de significar o que ele deveria
> significar — e ninguém, olhando o notebook, vai saber que deixou.

## Parte 3 — Explorar (só o treino)

Duas perguntas, dois gráficos.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))

caros = treino[treino["caro"] == 1]
baratos = treino[treino["caro"] == 0]
eixos[0].scatter(baratos["longitude"], baratos["latitude"], s=3,
                 c="#B0B0B0", alpha=0.35, label="não caro")
eixos[0].scatter(caros["longitude"], caros["latitude"], s=3,
                 c="#0F5028", alpha=0.45, label="caro")
eixos[0].set_xlabel("longitude"); eixos[0].set_ylabel("latitude")
eixos[0].set_title("Onde estão os distritos caros")
eixos[0].legend(markerscale=4)

taxa = treino.groupby("ocean_proximity")["caro"].mean().sort_values()
eixos[1].barh(taxa.index, taxa.values, color="#0F5028")
eixos[1].set_xlabel("proporção de distritos caros")
eixos[1].set_title("Por proximidade do oceano")
eixos[1].grid(axis="x", alpha=0.3)

plt.tight_layout(); plt.show()

print(treino.groupby("ocean_proximity")["caro"].agg(["mean", "size"]).round(4))

**O que os dois gráficos dizem.**

O da esquerda é o mapa da Califórnia, desenhado sem querer: os distritos caros
formam duas manchas, a baía de São Francisco e a faixa de Los Angeles. **Latitude e
longitude, sozinhas, já carregam muita informação** — e isso vai aparecer no
desempenho do modelo.

O da direita mostra o abismo: perto da água a proporção de caros vai de **31%**
(`<1H OCEAN`) a **41%** (`NEAR BAY`); em `INLAND`, **4,5%**. Uma única coluna de
texto vale mais que várias numéricas.

> ⚠️ E repare no `ISLAND`: **100% de caros, em quatro distritos.** (São cinco no
> conjunto inteiro; quatro caíram no treino.) É uma proporção que não significa nada —
> quatro exemplos. Guarde isso para a Parte 5, quando o `OneHotEncoder` criar uma
> coluna inteira só para eles.

## Parte 4 — A linha de base burra

Antes de treinar qualquer coisa: **qual é a acurácia do modelo mais burro possível?**

O modelo mais burro aqui responde sempre "não é caro", sem olhar para nada.

In [ ]:
# TODO 3: calcule a acurácia e a revocação (recall) do modelo que responde
#         SEMPRE 0. Pense antes de escrever: quantos ele acerta? E dos caros,
#         quantos ele encontra?
acuracia_base = 0.0
recall_base = 0.0

print(f"acurácia do 'sempre não caro' : {acuracia_base:.4f}")
print(f"recall   do 'sempre não caro' : {recall_base:.4f}")
print( "esperado                      : 0.7511  e  0.0000")

**Guarde este número: 75,1%.**

Qualquer modelo que você treinar hoje tem que ser comparado com ele. Um classificador
com 80% de acurácia neste problema não é "bom": é **cinco pontos melhor que não fazer
nada**.

E o recall zero é o ponto mais importante da aula passada, agora em dados reais: o
modelo burro tem acurácia alta **e não encontra um único distrito caro**. Se o seu
objetivo for justamente achar os caros, ele é inútil, com 75% de acerto.

> 📖 É o argumento do Géron no cap. 3 (p. 87): num problema desbalanceado *"a acurácia
> geralmente não é a medida de desempenho preferida para classificadores"*.

## Parte 5 — Preparar os dados

Três problemas, três ferramentas:

| Problema | Ferramenta |
|---|---|
| 207 valores faltando em `total_bedrooms` | `SimpleImputer(strategy="median")` |
| `ocean_proximity` é texto | `OneHotEncoder` |
| escalas de 0,5 a 39.320 na mesma tabela | `StandardScaler` |

E um quarto problema, que é de engenharia: aplicar isso **na ordem certa, só nas
colunas certas, e do mesmo jeito no treino e no teste**. A peça que faz isso é o
`ColumnTransformer`.

> ⚠️ **A mediana sai só do treino.** É a mesma regra do `StandardScaler` no lab04, e
> vale igual para o imputador: `fit` no treino, `transform` nos dois. Quem calcula a
> mediana com o conjunto inteiro vazou o teste sem perceber.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer          # NÃO é sklearn.preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM = ["longitude", "latitude", "housing_median_age", "total_rooms",
       "total_bedrooms", "population", "households", "median_income"]
CAT = ["ocean_proximity"]

# TODO 4: monte o ColumnTransformer.
#   - nas colunas NUM: um Pipeline com SimpleImputer(strategy="median")
#     seguido de StandardScaler()
#   - nas colunas CAT: OneHotEncoder(handle_unknown="ignore")
#   Dica: ColumnTransformer([("num", <transformador>, NUM),
#                            ("cat", <transformador>, CAT)])
#   (o que está abaixo é provisório: trata as numéricas e JOGA FORA o texto.
#    Roda, mas a saída tem 8 colunas em vez de 13 — falta o segundo ramo.)
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("esc", StandardScaler())]), NUM),
], remainder="drop")

X_treino = treino[NUM + CAT]
y_treino = treino["caro"].values
X_teste  = teste[NUM + CAT]
y_teste  = teste["caro"].values

saida = prep.fit_transform(X_treino)
print(f"entrada : {len(NUM)} colunas numéricas + {len(CAT)} de texto")
print(f"saída   : {saida.shape[1]} colunas   (esperado: 13)")
print(f"faltantes na saída: {np.isnan(np.asarray(saida, dtype=float)).sum()}")

**8 + 5 = 13.** As oito numéricas continuam oito; a coluna de texto virou cinco,
uma por valor possível — inclusive uma coluna inteira dedicada aos **cinco** distritos
`ISLAND`.

E o `handle_unknown="ignore"` é o que impede o notebook de quebrar se aparecer, no
teste, uma categoria que não existia no treino. Com 5 exemplos de `ISLAND` em 20 mil,
isso é uma possibilidade real: bastava a divisão ter mandado os cinco para o teste.

## Parte 6 — A armadilha do vazamento

Antes de treinar o modelo de verdade, um experimento. Vamos deixar a coluna
`median_house_value` **entre as entradas** — por distração, como acontece de verdade
quando alguém escreve `X = dados.drop(columns=["caro"])` e segue em frente.

Rode e olhe o número.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

VAZA = NUM + ["median_house_value"]
prep_vazado = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("esc", StandardScaler())]), VAZA),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
modelo_vazado = Pipeline([
    ("prep", prep_vazado),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEMENTE)),
])

nota = cross_val_score(modelo_vazado, treino[VAZA + CAT], y_treino,
                       cv=5, scoring="accuracy")
print(f"acurácia em validação cruzada: {nota.mean():.4f}   (esperado: 0.9981)")

**99,81%.** E está errado.

`median_house_value` **é** o rótulo. A coluna `caro` foi construída a partir dela —
o modelo não aprendeu nada sobre a Califórnia, só redescobriu a comparação
`> 265000`. Não chega a 100% porque a fronteira é uma reta no espaço padronizado e o
corte é exato, então sobram alguns casos na borda.

> 🚩 **A regra prática:** resultado bom demais é suspeito, e a primeira coisa a fazer
> é procurar, entre as entradas, alguma que só existiria **depois** de saber a
> resposta. Aqui é óbvio porque a coluna tem o nome do alvo. Nos casos que aparecem
> na vida real ela quase nunca tem — é um "número de protocolo" que só é preenchido
> quando o processo termina, uma data de cancelamento, um campo que o sistema
> preenche depois.

Agora, sem a coluna, para valer.

## Parte 7 — Regressão logística, e a matriz de confusão

A regressão logística é a irmã classificadora da regressão linear do lab04: mesma
combinação $b_0 + b_1x_1 + \dots$, passada por uma função que espreme o resultado
para o intervalo $(0,1)$, e lida como probabilidade.

Vamos avaliá-la com **validação cruzada em 5 dobras**, no treino. Nada de teste ainda.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

logistica = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEMENTE)),
])

nota_log = cross_val_score(logistica, X_treino, y_treino, cv=5, scoring="accuracy")
print(f"acurácia CV: {nota_log.mean():.4f}  (± {nota_log.std():.4f})   "
      f"(esperado: 0.8786 ± 0.0056)")
print(f"por dobra  : {np.round(nota_log, 4)}")
print(f"\nlinha de base burra: {acuracia_base:.4f}")

**87,9% contra 75,1%.** O modelo aprendeu alguma coisa — doze pontos acima do
burro. Mas a acurácia sozinha não diz **que tipo** de erro ele comete. Para isso,
a matriz.

In [ ]:
pred_log = cross_val_predict(logistica, X_treino, y_treino, cv=5)
m = confusion_matrix(y_treino, pred_log)
(vn, fp), (fn, vp) = m

print("                   previsto")
print("                não caro    caro")
print(f"real  não caro   {vn:8d}{fp:8d}")
print(f"      caro       {fn:8d}{vp:8d}")
print(f"\nesperado:  11720  682  /  1322  2788")

In [ ]:
# TODO 5: calcule as três métricas a partir de vn, fp, fn, vp.
#   precisão = dos que o modelo chamou de caros, quantos eram
#   recall   = dos que eram caros, quantos o modelo achou
#   F1       = 2 * precisão * recall / (precisão + recall)
precisao = 0.0
recall = 0.0
f1 = 0.0

print(f"precisão : {precisao:.4f}   (esperado: 0.8035)")
print(f"recall   : {recall:.4f}   (esperado: 0.6783)")
print(f"F1       : {f1:.4f}   (esperado: 0.7356)")

**Leia a matriz, não a acurácia.**

Dos 4.110 distritos caros, o modelo encontrou 2.788 e **deixou escapar 1.322** —
quase um terço. Quando ele diz "caro", acerta em 80% das vezes; mas ele diz "caro"
pouco.

Esse desequilíbrio entre precisão (0,80) e recall (0,68) não é acidente: o modelo foi
treinado para maximizar acerto total, e com três quartos de "não caro" na base, chutar
"não caro" na dúvida é o que mais rende. **A base desbalanceada empurra o modelo para
a classe maior.**

## Parte 8 — A árvore, e o sobreajuste em estado puro

Troque o classificador, mantenha o resto. Primeiro, a nota **no treino**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arvore = Pipeline([
    ("prep", prep),
    ("clf", DecisionTreeClassifier(random_state=SEMENTE)),
])
arvore.fit(X_treino, y_treino)

print(f"acurácia NO TREINO: {arvore.score(X_treino, y_treino):.4f}")

**1,0000. Acurácia perfeita.**

Não comemore. Uma árvore sem limite de profundidade cresce até separar cada exemplo
do treino — no limite, uma folha por distrito. Ela não aprendeu a Califórnia; ela
**decorou** 16.512 linhas.

E é por isso que "avaliar no treino" não é uma avaliação. Rode a validação cruzada e
compare.

In [ ]:
nota_arv = cross_val_score(arvore, X_treino, y_treino, cv=5, scoring="accuracy")
print(f"acurácia NO TREINO           : 1.0000")
print(f"acurácia em validação cruzada: {nota_arv.mean():.4f}  "
      f"(± {nota_arv.std():.4f})   (esperado: 0.8708 ± 0.0053)")
print(f"\nlogística                    : {nota_log.mean():.4f}")

**De 1,0000 para 0,8708.** A distância entre esses dois números *é* o sobreajuste,
medido. E a árvore, apesar da nota perfeita no treino, fica **abaixo** da logística
quando avaliada em dado que ela não viu.

> 💡 A validação cruzada não melhora o modelo. Ela só impede você de acreditar num
> número que não significa nada.

## Parte 9 — A floresta

Uma árvore decora. Cem árvores, cada uma treinada num pedaço diferente dos dados e
com um subconjunto diferente das colunas, decoram coisas diferentes — e a votação
entre elas cancela boa parte do decorado. É a ideia dos **métodos de ensemble**
(Géron, p. 78).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO 6: monte a floresta no mesmo molde dos dois anteriores: um Pipeline com
#         ("prep", prep) e ("clf", RandomForestClassifier(...)).
#         Use n_estimators=100, random_state=SEMENTE e n_jobs=-1.
floresta = logistica   # provisório

nota_flo = cross_val_score(floresta, X_treino, y_treino, cv=5, scoring="accuracy")
print(f"floresta  : {nota_flo.mean():.4f}  (± {nota_flo.std():.4f})   "
      f"(esperado: 0.9071 ± 0.0022)")
print(f"árvore    : {nota_arv.mean():.4f}")
print(f"logística : {nota_log.mean():.4f}")
print(f"base burra: {acuracia_base:.4f}")

**0,9071.** Três pontos acima da logística, e o desvio entre as dobras caiu para
0,0022 — além de melhor, é mais estável.

Vamos olhar a matriz dela.

In [ ]:
prob_flo = cross_val_predict(floresta, X_treino, y_treino, cv=5,
                            method="predict_proba")[:, 1]
pred_flo = (prob_flo >= 0.5).astype(int)
(vn, fp), (fn, vp) = confusion_matrix(y_treino, pred_flo)

print("                   previsto")
print("                não caro    caro")
print(f"real  não caro   {vn:8d}{fp:8d}")
print(f"      caro       {fn:8d}{vp:8d}")
print(f"\nesperado:  11889  513  /  1022  3088")
print(f"\nprecisão {vp/(vp+fp):.4f}   recall {vp/(vp+fn):.4f}   "
      f"F1 {2*vp/(2*vp+fp+fn):.4f}")
print( "esperado 0.8575          0.7513          0.8009")

Comparando com a logística: os falsos negativos caíram de 1.322 para 1.022, e os
falsos positivos de 682 para 513. **Melhorou nos dois lados**, o que nem sempre
acontece.

> 📌 **Por que `predict_proba` e não `predict`.** Pedimos a probabilidade e aplicamos
> o corte de 0,5 nós mesmos. O resultado é o mesmo do `predict` — e é exatamente o que
> a próxima parte precisa.

## Parte 10 — O limiar

O classificador não decide "caro" ou "não caro". Ele produz uma **probabilidade**, e
alguém compara com 0,5.

Esse 0,5 não vem de lugar nenhum. É uma escolha, e é **sua**.

In [ ]:
# TODO 7: para cada limiar da lista, transforme prob_flo em previsão 0/1 e
#         calcule precisão, recall e acurácia.
#         Dica: previsto = (prob_flo >= limiar).astype(int)
print(f"{'limiar':>7} {'precisão':>10} {'recall':>8} {'acurácia':>10}")
for limiar in [0.30, 0.50, 0.70]:
    previsto = np.zeros_like(y_treino)     # TODO: troque esta linha
    (vn, fp), (fn, vp) = confusion_matrix(y_treino, previsto)
    p = vp / (vp + fp) if (vp + fp) else 0.0
    r = vp / (vp + fn) if (vp + fn) else 0.0
    print(f"{limiar:7.2f} {p:10.4f} {r:8.4f} {(vn+vp)/len(y_treino):10.4f}")

print("\nesperado:")
print("   0.30     0.7264   0.8786     0.8874")
print("   0.50     0.8575   0.7513     0.9070")
print("   0.70     0.9476   0.5852     0.8887")

**Leia a tabela de cima para baixo.** Precisão sobe de 0,73 para 0,95; recall cai
de 0,88 para 0,59. Não existe limiar que melhore os dois: é a ***compensação
precisão/revocação*** do cap. 3 (p. 91), e ela é estrutural, não um defeito do modelo.

**Quem escolhe é o problema, não o algoritmo.** Dois exemplos com o mesmo modelo:

- uma corretora que quer **uma lista curta e confiável** de bairros caros para
  abordar → limiar alto, 0,70: quase tudo que aparece na lista está certo;
- uma prefeitura mapeando **onde o preço da moradia está fora do alcance** da
  população → limiar baixo, 0,30: deixar um bairro de fora é pior que investigar um
  a mais.

E note que o limiar de **melhor acurácia** é 0,50, que não é o melhor para nenhum dos
dois. **A métrica que você otimiza é uma decisão de negócio.**

In [ ]:
from sklearn.metrics import precision_recall_curve

p, r, cortes = precision_recall_curve(y_treino, prob_flo)

plt.figure(figsize=(6.5, 4))
plt.plot(cortes, p[:-1], color="#0F5028", lw=2, label="precisão")
plt.plot(cortes, r[:-1], color="#B02020", lw=2, ls="--", label="recall")
plt.axvline(0.5, color="#7A7A7A", ls=":", lw=1.5)
plt.xlabel("limiar"); plt.ylabel("valor da métrica")
plt.title("Uma sobe, a outra desce. Sempre.")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Parte 11 — A avaliação final

Agora, e só agora, o conjunto de teste.

**Uma vez.** Não é figura de linguagem: se você olhar o resultado no teste, achar
ruim, mudar alguma coisa e rodar de novo, o teste virou um conjunto de validação caro
— e você não tem mais nenhuma estimativa honesta de como o modelo vai se sair no
mundo.

In [ ]:
# TODO 8: treine a floresta em TODO o treino e avalie no teste, uma vez.
#         Dica: floresta.fit(X_treino, y_treino)
#               pred_teste = floresta.predict(X_teste)
pred_teste = np.zeros_like(y_teste)

(vn, fp), (fn, vp) = confusion_matrix(y_teste, pred_teste)
print("                   previsto")
print("                não caro    caro")
print(f"real  não caro   {vn:8d}{fp:8d}")
print(f"      caro       {fn:8d}{vp:8d}")
print(f"\nacurácia : {(vn+vp)/len(y_teste):.4f}   (esperado: 0.9123)")
print(f"precisão : {vp/(vp+fp) if (vp+fp) else 0:.4f}   (esperado: 0.8713)")
print(f"recall   : {vp/(vp+fn) if (vp+fn) else 0:.4f}   (esperado: 0.7637)")
print(f"\nlinha de base no teste: {1-teste['caro'].mean():.4f}")

**0,9123 no teste, contra 0,9071 na validação cruzada.**

Os dois números batem — e é isso que se espera quando nada vazou. Um teste muito
**melhor** que a validação cruzada é tão suspeito quanto um muito pior: costuma
significar que o teste não é representativo, ou que alguma informação dele chegou ao
treino por um caminho que ninguém viu.

E o número que fecha a aula: **91,2% contra 74,9% da linha de base**. Dezesseis pontos.
É esse o valor do modelo, e é assim que ele se reporta — nunca "91% de acurácia" solto,
sempre "91% contra 75% de não fazer nada".

## Parte 12 — Análise de erros

O modelo errou 362 distritos no teste. **Onde?**

In [ ]:
teste_erros = teste.copy()
teste_erros["previsto"] = pred_teste
teste_erros["errou"] = (teste_erros["previsto"] != teste_erros["caro"]).astype(int)

print("taxa de erro por proximidade do oceano:")
print(teste_erros.groupby("ocean_proximity")["errou"].agg(["mean", "size"]).round(4))

print("\ndistância do valor real ao corte, nos acertos e nos erros:")
d_corte = (teste_erros["median_house_value"] - CORTE).abs()
print(f"  acertou : mediana {reais(d_corte[teste_erros['errou'] == 0].median())}")
print(f"  errou   : mediana {reais(d_corte[teste_erros['errou'] == 1].median())}")

**O modelo erra perto da fronteira.** Nos distritos que ele acertou, o valor real
está, na mediana, a **R$ 115.000** do corte; nos que ele errou, a **R$ 29.850**.
Quatro vezes mais perto.

E a taxa de erro por região confirma a mesma coisa por outro ângulo: `INLAND` é a
região com **menos erro** (3,7%), e é justamente a que quase não tem distritos caros —
lá o modelo raramente precisa decidir. `NEAR OCEAN`, com 38% de caros, é a de **mais
erro** (14,4%): é onde ficam as decisões difíceis.

Isso não é defeito: é consequência de ter transformado um número contínuo em duas
caixas. Um distrito de 264 mil e um de 266 mil são, no mundo, praticamente iguais — e
recebem rótulos opostos. **Nenhum modelo vai separar bem os dois**, porque não há o
que separar.

> 💡 É o que a **análise de erros** (Géron, cap. 3, p. 100) serve para descobrir. Aqui
> ela diz que o caminho para melhorar não passa por trocar de algoritmo: passa por
> rever o enquadramento — três faixas em vez de duas, ou uma zona de "não sei" perto
> do corte.

## Parte 13 — Desafio (opcional)

Sem dica e sem TODO. Escolha um:

**(a)** Remova `longitude` e `latitude` das entradas e refaça a validação cruzada da
floresta. Quanto a localização, sozinha, estava valendo?

**(b)** Troque o corte de 265.000 para a **mediana** do `median_house_value`. O
problema fica balanceado. O que acontece com a acurácia, com a linha de base, e com a
distância entre as duas?

**(c)** Use `class_weight="balanced"` na regressão logística e compare a matriz de
confusão com a da Parte 7. Qual das quatro casas mudou mais?

---

## Antes de sair do laboratório

As máquinas do laboratório são restauradas ao reiniciar. **Salve o notebook na
nuvem** (OneDrive institucional, Google Drive) ou envie para você mesmo — o arquivo
não sobrevive ao próximo boot.

E apague qualquer login que tenha feito no navegador.

---

## O que este laboratório mostrou

1. **Separe antes de olhar.** Explorar o conjunto inteiro e depois separar já é
   vazamento — o seu, não o do código.
2. **A linha de base vem primeiro.** 75,1% de acurácia sem fazer nada. Todo número
   depois disso se lê contra ele.
3. **Acurácia alta com recall zero existe**, e é o caso mais comum em base
   desbalanceada.
4. **Resultado bom demais é defeito, não conquista.** 99,81% foi o rótulo entrando
   pela porta dos fundos.
5. **1,0000 no treino é sobreajuste**, e a validação cruzada é o que denuncia.
6. **O limiar é decisão sua.** Precisão e recall se movem em sentidos opostos, e quem
   escolhe o ponto é o problema.
7. **O teste, uma vez.** Depois de olhar, ele deixa de ser teste.
8. **Os erros ficam na fronteira**, porque a fronteira foi inventada por nós.